# Minimal LoRA XLM-R Baseline

Very small cluster notebook for trying a transformer baseline with LoRA. The model is `xlm-roberta-base`, i.e. XLM-R base; there is no standard Hugging Face model called `xlm-bert-base`.

This mirrors the useful part of `current_sota.ipynb`: train a multilingual regression model for ratings `0..4`, evaluate rounded MAE, and optionally write a submission.

In [ ]:
# Run this once on the cluster if the environment is missing packages.
# %pip install -q -U "transformers>=4.40" "datasets" "accelerate" "peft>=0.10" "scikit-learn"

In [ ]:
from pathlib import Path
import inspect
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, Value
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, Trainer, TrainingArguments, set_seed

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from experiments.config import ModelConfig
from experiments.models import SentimentModel

SEED = 42
MODEL_ID = "xlm-roberta-base"
TRAIN_CSV = ROOT / "data" / "train_lang.csv"
TEST_CSV = ROOT / "data" / "test.csv"
OUTPUT_DIR = ROOT / "outputs" / "minimal_lora_xlmr"

# Use a small number like 20000 for a smoke test; set to None for full training.
SAMPLE_N = None
VAL_SIZE = 0.10
MAX_LENGTH = 128

BATCH_SIZE = 64
EVAL_BATCH_SIZE = 256
EPOCHS = 3
LR = 1.5e-4
FP16 = torch.cuda.is_available()

os.environ.setdefault("WANDB_DISABLED", "true")
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Data

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df["sentence"] = df["sentence"].fillna("")

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE_N, random_state=SEED, stratify=df["label"])

train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SEED, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train", train_df.shape, "val", val_df.shape)
print(train_df["label"].value_counts().sort_index().to_dict())
train_df.head()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize(batch):
    out = tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    out["labels"] = [float(x) for x in batch["label"]]
    if "lang" in batch:
        out["lang"] = [0 if x == "eng_Latn" else 1 for x in batch["lang"]]
    return out

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)
remove_cols = train_ds.column_names

train_ds = train_ds.map(tokenize, batched=True, remove_columns=remove_cols)
val_ds = val_ds.map(tokenize, batched=True, remove_columns=remove_cols)
train_ds = train_ds.cast_column("labels", Value("float32"))
val_ds = val_ds.cast_column("labels", Value("float32"))
if "lang" in train_ds.column_names:
    train_ds = train_ds.cast_column("lang", Value("int64"))
    val_ds = val_ds.cast_column("lang", Value("int64"))
train_ds.set_format("torch")
val_ds.set_format("torch")

## Model

In [ ]:
model_config = ModelConfig(
    kind="lora_bert",
    name=MODEL_ID,
    geometry="default",
    lora_r=128,
    lora_alpha=64,
    lora_dropout=0.01,
)
model = SentimentModel.from_config(model_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## Train

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.asarray(logits).reshape(-1)
    labels = np.asarray(labels).reshape(-1)
    rounded = np.rint(np.clip(preds, 0, 4))
    return {
        "mae": float(mean_absolute_error(labels, preds)),
        "rounded_mae": float(mean_absolute_error(labels, rounded)),
    }


def make_training_args(**kwargs):
    # Transformers renamed evaluation_strategy -> eval_strategy in recent versions.
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    return TrainingArguments(**kwargs)


args = make_training_args(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    overwrite_output_dir=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=EPOCHS,
    evaluation_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=FP16,
    report_to=[],
    remove_unused_columns=False,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
metrics

In [ ]:
final_dir = OUTPUT_DIR / "final_model"
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(final_dir)

## Optional submission

In [ ]:
if TEST_CSV.exists():
    test_df = pd.read_csv(TEST_CSV)
    test_df["sentence"] = test_df["sentence"].fillna("")

    test_ds = Dataset.from_pandas(test_df, preserve_index=False)

    def tokenize_test(batch):
        return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    test_tok = test_ds.map(tokenize_test, batched=True, remove_columns=test_ds.column_names)
    test_tok.set_format("torch")

    raw = trainer.predict(test_tok).predictions.reshape(-1)
    preds = np.rint(np.clip(raw, 0, 4)).astype(int)

    submission = pd.DataFrame({"id": test_df["id"], "label": preds})
    submission_path = OUTPUT_DIR / "submission.csv"
    submission_path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(submission_path, index=False)
    print(submission_path)
    display(submission.head())
else:
    print("No test CSV found:", TEST_CSV)